# Likelihood & maximum likelihood

_Machine Learning_

**Pick the parameters that make your data look most probable.**

The likelihood asks: given some parameters, how probable is the data we actually saw?

     Different parameters make the data more or less likely.

---

This notebook is a practice scaffold for the **Likelihood & maximum likelihood** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — NumPy

### Step 1 — State the observed data and the likelihood

We observed **7 heads in 10 coin flips**. The likelihood of a heads-probability `theta` is `theta^h · (1-theta)^(n-h)`. Working in **log space** turns that product into a sum, which is easier and more numerically stable to compute. Maximising the log-likelihood gives the same answer as maximising the likelihood.

In [ ]:
import numpy as np

# observed: 7 heads in 10 flips. Which heads-probability best explains it?
h, n = 7, 10

def log_likelihood(theta):
    # log of theta^h (1-theta)^(n-h)
    return h * np.log(theta) + (n - h) * np.log(1 - theta)

### Step 2 — Search a grid of theta values

We sweep `theta` across a fine grid from 0.01 to 0.99 and evaluate the log-likelihood at each point. The grid value with the highest log-likelihood is our numerical estimate of the best parameter. (We avoid 0 and 1 exactly because `log(0)` is undefined.)

In [ ]:
thetas = np.linspace(0.01, 0.99, 99)
ll = log_likelihood(thetas)
best = thetas[np.argmax(ll)]

### Step 3 — Compare the grid search to the closed-form answer

For a coin, calculus gives a tidy closed-form maximum-likelihood estimate: `theta = h/n`. We print the log-likelihood at two candidate values, the grid's argmax, and the closed-form `h/n` — they should agree, confirming the grid search found the true peak at 0.7.

In [ ]:
print("log-likelihood at 0.5:", round(float(log_likelihood(0.5)), 4))
print("log-likelihood at 0.7:", round(float(log_likelihood(0.7)), 4))
print("argmax theta (grid):", round(float(best), 2))
print("closed-form MLE h/n :", h / n)

## Visualize the data & results

_Of 569 real tumors, 212 are malignant. Which malignancy rate theta best explains that count?_

### Step 1 — Count successes in the real dataset

Now we apply the same idea to real data. Among the 569 breast-cancer tumours, we count how many are malignant — that count plays the role of `h` (successes), and the total plays the role of `n`. The question becomes: which malignancy rate `theta` best explains that count?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

# real data: count malignant tumors among all 569
bc = load_breast_cancer()
h = int((bc.target == 0).sum())    # malignant successes
n = len(bc.target)

### Step 2 — Evaluate the log-likelihood over a grid

We reuse the same log-likelihood formula, now with the real `h` and `n`, and evaluate it across a grid of candidate malignancy rates. The curve will peak at the rate that makes the observed count most probable — which is just `h/n`, the fraction of tumours that are malignant.

In [ ]:
def log_likelihood(theta):
    return h * np.log(theta) + (n - h) * np.log(1 - theta)

thetas = np.linspace(0.05, 0.95, 25)
ll = log_likelihood(thetas)

### Step 3 — Plot the log-likelihood curve

Plotting log-likelihood against `theta` shows a single hump. The top of the hump is the maximum-likelihood estimate — the malignancy rate the data points to most strongly.

In [ ]:
plt.plot(thetas, ll, color="#c89bff", label="log L(theta)")
plt.xlabel("theta (malignancy rate)")
plt.ylabel("log-likelihood")
plt.title("Log-likelihood vs malignancy rate theta")
plt.legend()
plt.show()